In [1]:
import tensorflow as tf
tf.enable_eager_execution()
tf.executing_eagerly()
import warnings
warnings.filterwarnings('ignore') 

print("*"*100)

import logging
import os
import numpy as np
import requests
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import *
from tensorflow.python.keras.layers import Layer
from tensorflow.keras import regularizers

from aibrain_common.conf import tf_context
from aibrain_common.data.dataset_builder import DatasetBuilder
from aibrain_common.utils import archive_utils, env_utils, oss_utils
from aibrain_job.utils import param_utils
from aibrain_common.utils.date_convert_utils import DateConvertUtils
from features_info import get_feature_columns
from spark_session_utils import SparkSessionHelper

from config import train_data_df
from features_info import get_features_info
from features_info import get_feature_columns
from aibrain_common.utils import date_convert_utils
from aibrain_common.component import tools

from metric import auc, gauc, labels_mean, logits_mean, topk_acc, mse

from ple import PleLayer # ple模型
from fm import FM # FM模型


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

****************************************************************************************************


### get_df_iterator函数

In [30]:
def get_df_iterator(input_table, partitions, batch_size):
    dataset_builder = DatasetBuilder(input_table=input_table, partitions=partitions)
    hdfs_reader = dataset_builder.get_reader()
    tf_env = env_utils.prepare_tf_env()
    logger.warning(f'tf_env: {tf_env}')
    num_worker = tf_env['WORKER_SIZE']
    index_worker = tf_env['TASK_ID']
    df_iterator = hdfs_reader.to_iterator_for_pdf(
                num_worker=num_worker, 
                index_worker=index_worker, 
                batch_size=batch_size,
                shuffle=False, 
                columns=[i.feature_name for i in get_features_info()] + ['click_label'] + ['order_label'] + ['gmv_label']
    )
    return df_iterator

In [45]:
input_table = 'turing_dev.rent_cars_all_feature_old_new_model_train_data'
train_pt = '20250119'

partitions = [['pt=' + str(train_pt)]] # [['pt=20240709']]

In [6]:
# 创建一个 DatasetBuilder 类的实例，用于初始化和配置数据集的构建过程。
"""
DatasetBuilder 是一个类，通常用于构建和管理数据集。它的作用是：
    加载数据：根据指定的输入源（如文件路径、数据库表名等）加载数据。
    配置数据处理：设置数据的预处理方式，例如分区、分片、数据格式化等。
    提供数据读取接口：通过方法（如 get_reader()）返回一个数据读取器，用于后续的数据加载和处理。
    
    初始化一个 DatasetBuilder 实例，用于构建和管理数据集。
    通过传入的 input_table 参数指定数据的来源。
    通过 partitions 参数配置数据的分区信息。
    为后续的数据加载和处理提供一个基础配置。
"""
dataset_builder = DatasetBuilder(input_table=input_table, partitions=partitions)

In [7]:
print(dataset_builder)

In [8]:
# 这行代码的作用是通过 dataset_builder 的 get_reader() 方法获取一个数据读取器对象，并将其赋值给变量 
"""
get_reader() 是 DatasetBuilder 类的一个方法，其主要作用是根据 dataset_builder 的配置信息（如数据源路径和分区信息）创建一个数据读取器对象。
"""
hdfs_reader = dataset_builder.get_reader()
hdfs_reader

In [9]:
# 初始化 TensorFlow 的运行环境，并从中提取与分布式训练相关的配置信息。

# 从 env_utils 模块中调用 prepare_tf_env() 方法，用于初始化 TensorFlow 的运行环境，特别是为分布式训练做准备。
# tf_env 是一个字典，包含了与 TensorFlow 运行环境相关的配置信息，例如分布式训练的参数。
tf_env = env_utils.prepare_tf_env()
# 将 tf_env 的内容输出到日志中，方便开发者调试和监控环境配置。
logger.warning(f'tf_env: {tf_env}')
# num_worker 表示分布式训练中工作节点（worker）的总数。在 TensorFlow 的分布式训练中，多个工作节点可以并行处理数据和训练模型。
num_worker = tf_env['WORKER_SIZE']
# 在分布式训练中，每个工作节点都有一个唯一的索引，用于标识其在集群中的位置。
index_worker = tf_env['TASK_ID']

2025-01-22 11:41:48,775 - __main__ - WARNING - tf_env: {'WORKER_SIZE': 1, 'TASK_SIZE': 1, 'JOB_NAME': 'worker', 'TASK_ID': 0}


In [10]:
tf_env
# 'WORKER_SIZE': 1  表示分布式训练中工作节点（worker）的总数为 1。意味着当前环境没有启用分布式训练，或者虽然配置了分布式环境，但实际上只有一个节点在运行。
# 'TASK_SIZE': 1   表示当前任务的总数为 1。与 WORKER_SIZE 为 1 是一致的，进一步说明当前环境可能没有启用分布式训练。
# 'JOB_NAME': 'worker'  表示当前任务的名称为 'worker'。在 TensorFlow 的分布式训练中，任务通常分为几种类型，如 worker、chief、ps（参数服务器）等。 'worker' 表示当前任务是一个工作节点，负责执行训练任务。 如果环境中只有一个工作节点（WORKER_SIZE=1），那么这个节点既是唯一的 worker，也可能承担了所有任务。
# 'TASK_ID': 0  表示当前任务的索引为 0。在只有一个工作节点的情况下，TASK_ID=0 表示这是第一个（也是唯一一个）任务。

{'WORKER_SIZE': 1, 'TASK_SIZE': 1, 'JOB_NAME': 'worker', 'TASK_ID': 0}

In [11]:
batch_size = 1024

In [46]:
# 作用是通过 hdfs_reader 创建一个数据迭代器 df_iterator，用于从 HDFS 中逐批读取数据，并将其组织成适合后续处理（如模型训练）的格式。

"""
to_iterator_for_pdf:
    hdfs_reader 是之前通过 dataset_builder.get_reader() 获取的数据读取器对象，专门用于从 HDFS 中读取数据。
    to_iterator_for_pdf() 是 hdfs_reader 提供的一个方法，用于将 HDFS 中的数据转换为一个迭代器（df_iterator）。这个迭代器会逐批返回数据，格式可能是 Pandas DataFrame（pdf 可能是 Pandas DataFrame 的缩写）。

参数:
    num_worker&index_worker 这两个参数用于分布式训练场景，控制数据的分片和分配。
        num_worker：工作节点的总数。
        index_worker：当前工作节点的索引。
    batch_size：指定每次迭代返回的数据批次大小。例如，如果 batch_size=1024，则每次返回 1024 条数据。
    shuffle=False：指定是否对数据进行随机打乱。这里设置为 False，表示数据不会被随机打乱。这可能是为了保留数据的原始顺序，或者是因为数据已经在加载前被随机化过了。
    columns：指定需要从 HDFS 数据中读取的列名称。列表的内容由特征和标签组成。（确保迭代器只加载指定的列，减少内存占用并提高数据处理效率。）
"""
df_iterator = hdfs_reader.to_iterator_for_pdf(
            num_worker=num_worker, 
            index_worker=index_worker, 
            batch_size=batch_size,
            shuffle=False, 
            columns=[i.feature_name for i in get_features_info()] + ['click_label'] + ['order_label'] + ['gmv_label']
)

In [25]:
"""
df_iterator 是一个迭代器对象，能够逐批返回数据。
每次迭代返回的数据是一个 Pandas DataFrame，包含指定的列（特征列和标签列）。
这种格式便于后续的数据处理、特征工程和模型训练。
"""

'\ndf_iterator 是一个迭代器对象，能够逐批返回数据。\n每次迭代返回的数据是一个 Pandas DataFrame，包含指定的列（特征列和标签列）。\n这种格式便于后续的数据处理、特征工程和模型训练。\n'

In [26]:
df_iterator

<generator object HdfsReader.to_iterator_for_pdf at 0x7fa3b78bb050>

### generator函数

In [59]:
def generator(input_table, partitions, batch_size):
    df_iterator = get_df_iterator(input_table, partitions, batch_size)
    for pandas_df in df_iterator:
        click_labels =  np.expand_dims(pandas_df['click_label'].to_numpy(), axis=-1)
        order_labels = np.expand_dims(pandas_df['order_label'].to_numpy(), axis=-1)
        gmv_labels = np.expand_dims(pandas_df['gmv_label'].to_numpy(), axis=-1)
        
        pandas_df = pandas_df.drop(columns=['click_label','order_label','gmv_label']).applymap(lambda x: [x])
        features = pandas_df.to_dict('list')
        yield features, {'click_labels': click_labels, 'order_labels': order_labels, 'gmv_labels':gmv_labels}

In [57]:
# df_iterator = get_df_iterator(input_table, partitions, batch_size)
# for pandas_df in df_iterator: # DataFrame格式，里面是特征和label
#     print("111",pandas_df)

In [58]:
df_iterator = get_df_iterator(input_table, partitions, batch_size)
for pandas_df in df_iterator:
    click_labels_tmp = pandas_df['click_label'] # 一维的数据 (1024,)
    # print("123",click_labels_tmp.shape)
    """
    pandas_df['click_label'] 返回的是该列的所有数据，类型为 Pandas Series。
    .to_numpy() 这是 Pandas Series 的方法，用于将 Pandas Series 转换为 NumPy 数组。
    np.expand_dims(..., axis=-1)  np.expand_dims 是 NumPy 提供的一个函数，用于在指定的轴上增加一个新的维度。axis=-1：表示在数组的最后一个维度上增加一个新的轴。
    对于一维数组 [n_samples]，增加一个新的轴后，数组的形状会从 [n_samples] 变为 [n_samples, 1]。
    
    因为许多模型（如 TensorFlow 或 PyTorch）期望标签数据是一个二维数组（即每个样本的标签是一个向量）。
    """
    click_labels =  np.expand_dims(pandas_df['click_label'].to_numpy(), axis=-1) # 二维的list数据 (1024, 1)
    # print("456",click_labels.shape)
    order_labels = np.expand_dims(pandas_df['order_label'].to_numpy(), axis=-1)
    gmv_labels = np.expand_dims(pandas_df['gmv_label'].to_numpy(), axis=-1)
    
    pandas_df_tmp = pandas_df.drop(columns=['click_label','order_label','gmv_label']) # 去掉指定列，其余特征列，dataframe形式 比如licensetag：外牌
    # print("123",pandas_df_tmp.shape) # (1024, 193)
    
    """
    applymap(...)：这是 Pandas DataFrame 的方法，用于对 DataFrame 中的每个元素应用一个函数。
    在这里，使用了一个 lambda 匿名函数，将每个元素 x 转换为一个包含该元素的列表 [x]。
    例如，如果某个单元格的值是 1，转换后会变成 [1]；如果某个单元格的值是 'a'，转换后会变成 ['a']。
    """
    pandas_df = pandas_df.drop(columns=['click_label','order_label','gmv_label']).applymap(lambda x: [x]) # 去掉指定列，其余特征列，每个数据是一个list，licensetag：[外牌]
    # print("456",pandas_df.shape) # (1024, 193)
    # print("123",type(pandas_df)) # <class 'pandas.core.frame.DataFrame'>
    """
    to_dict()：这是 Pandas DataFrame 提供的一个方法，用于将 DataFrame 转换为字典。
    
    指定转换的方式。当参数为 'list' 时，每个键（列名）对应的值会是一个列表，包含该列的所有数据。其他可能的参数值包括 'dict'（返回嵌套字典）、'series'（返回列名到 Series 的映射）等。
    例如，如果 DataFrame 有两列 feature1 和 feature2，转换后的字典可能是：
    {
        'feature1': [1, 2, 3],
        'feature2': [4, 5, 6]
    }
    """
    features = pandas_df.to_dict('list')
    # print("456",type(features)) # <class 'dict'>
    # print("111",features)

W0122 13:34:30.031805 140342019123008 <ipython-input-42-60c8c733eed0>:5] tf_env: {'WORKER_SIZE': 1, 'TASK_SIZE': 1, 'JOB_NAME': 'worker', 'TASK_ID': 0}
I0122 13:34:30.043040 140342019123008 hdfs_reader.py:80] send http post
I0122 13:34:30.596238 140342019123008 hdfs_reader.py:88] send http post done: b'{"code":0,"msg":"\xe8\xa1\xa8\xe4\xbf\xa1\xe6\x81\xaf\xe6\x9f\xa5\xe8\xaf\xa2\xe6\x88\x90\xe5\x8a\x9f","data":{"":"serialization.null.format","usr_pub_age_level":"int","new_old_cross_group":"bigint","pt":"string","usr_pub_sex":"bigint","last_pay_diff_dt":"bigint","new_old_sex_age_consume_cross_group":"bigint","evaluate_add_info":"int","transmission_type":"int","new_old_age_level_cross_price":"bigint","CreateTime:":"Tue Dec 10 14:34:58 CST 2024","ave_price":"double","rentcars_orderdetails_cnt_14d":"bigint","Database:":"turing_dev          ","xl_sex_age_stu_yd_cross_price":"bigint","usr_pub_usual_active_city_lvl":"bigint","order_group_cnt":"bigint","zu_qi_deal":"int","homepage_pv":"bigint"

KeyboardInterrupt: 

### input_fn函数

In [61]:
def input_fn(input_table, partitions, num_epochs=None, batch_size=1024):
    dataset = tf.data.Dataset.from_generator(
        generator = lambda: generator(input_table, partitions, batch_size),
        output_types=(
            {i.feature_name: i.dtype for i in get_features_info()},
            {"click_labels": tf.float32 , "order_labels": tf.float32, "gmv_labels": tf.float32}
            
        ),
        output_shapes=(
            {i.feature_name: tf.TensorShape([None, 1]) for i in get_features_info()},
            {"click_labels": tf.TensorShape([None, 1]) , "order_labels": tf.TensorShape([None, 1]), "gmv_labels": tf.TensorShape([None, 1])}
        )
    )

    dataset = dataset.repeat(num_epochs)

    features, labels = dataset.make_one_shot_iterator().get_next()
    return features, labels

In [63]:
num_epochs=1

In [64]:
"""
这段代码的作用是构建一个 TensorFlow 数据集（tf.data.Dataset），从自定义生成器中加载数据，并将其配置为适合模型训练的格式。此外，它还设置了数据集的重复次数，并通过迭代器获取数据。
    从自定义生成器中创建一个 TensorFlow 数据集。
    配置数据集的输出类型和形状，使其适配模型训练的需求。
    设置数据集的重复次数（num_epochs）。
    创建单次迭代器并获取下一批数据，返回特征和标签。
    
例子：
    features = {
        'feature1': [[1], [2], [3]],
        'feature2': [[4], [5], [6]]
    }
    labels = {
        'click_labels': [[1.0], [0.0], [1.0]],
        'order_labels': [[0.0], [1.0], [0.0]],
        'gmv_labels': [[10.0], [20.0], [30.0]]
    }
"""

dataset = tf.data.Dataset.from_generator(
    generator = lambda: generator(input_table, partitions, batch_size),
    output_types=(
        {i.feature_name: i.dtype for i in get_features_info()},
        {"click_labels": tf.float32 , "order_labels": tf.float32, "gmv_labels": tf.float32}

    ),
    output_shapes=(
        {i.feature_name: tf.TensorShape([None, 1]) for i in get_features_info()},
        {"click_labels": tf.TensorShape([None, 1]) , "order_labels": tf.TensorShape([None, 1]), "gmv_labels": tf.TensorShape([None, 1])}
    )
)

# 设置数据集重复次数 
    # 如果 num_epochs 是一个整数，数据集会重复指定的次数。
    # 如果 num_epochs 是 None，数据集会无限重复，直到显式停止训练。
dataset = dataset.repeat(num_epochs) 

# 创建迭代器并获取数据
    # make_one_shot_iterator()：创建一个单次迭代器（one-shot iterator），它会在数据集耗尽时自动停止。
    # get_next()：从迭代器中获取下一批数据。
features, labels = dataset.make_one_shot_iterator().get_next()

W0122 14:46:43.274784 140291734492928 <ipython-input-42-60c8c733eed0>:5] tf_env: {'WORKER_SIZE': 1, 'TASK_SIZE': 1, 'JOB_NAME': 'worker', 'TASK_ID': 0}
I0122 14:46:43.278396 140291734492928 hdfs_reader.py:80] send http post
I0122 14:46:44.301595 140291734492928 hdfs_reader.py:88] send http post done: b'{"code":0,"msg":"\xe8\xa1\xa8\xe4\xbf\xa1\xe6\x81\xaf\xe6\x9f\xa5\xe8\xaf\xa2\xe6\x88\x90\xe5\x8a\x9f","data":{"":"serialization.null.format","usr_pub_age_level":"int","new_old_cross_group":"bigint","pt":"string","usr_pub_sex":"bigint","last_pay_diff_dt":"bigint","new_old_sex_age_consume_cross_group":"bigint","evaluate_add_info":"int","transmission_type":"int","new_old_age_level_cross_price":"bigint","CreateTime:":"Tue Dec 10 14:34:58 CST 2024","ave_price":"double","rentcars_orderdetails_cnt_14d":"bigint","Database:":"turing_dev          ","xl_sex_age_stu_yd_cross_price":"bigint","usr_pub_usual_active_city_lvl":"bigint","order_group_cnt":"bigint","zu_qi_deal":"int","homepage_pv":"bigint"

In [65]:
features

{'licensetag': <tf.Tensor: id=38841, shape=(1024, 1), dtype=string, numpy=
 array([[b'\xe5\xa4\x96\xe7\x89\x8c'],
        [b''],
        [b''],
        ...,
        [b''],
        [b''],
        [b'\xe5\xa4\x96\xe7\x89\x8c']], dtype=object)>,
 'platform': <tf.Tensor: id=38902, shape=(1024, 1), dtype=string, numpy=
 array([[b'mPaaS'],
        [b'mPaaS'],
        [b'Alipay'],
        ...,
        [b'Alipay'],
        [b'mPaaS'],
        [b'mPaaS']], dtype=object)>,
 'city_code': <tf.Tensor: id=38803, shape=(1024, 1), dtype=string, numpy=
 array([[b'0755'],
        [b'023'],
        [b'0898'],
        ...,
        [b'0898'],
        [b'0591'],
        [b'020']], dtype=object)>,
 'usr_pub_usual_active_city_code': <tf.Tensor: id=38976, shape=(1024, 1), dtype=string, numpy=
 array([[b'0755'],
        [b'023'],
        [b'0898'],
        ...,
        [b'0898'],
        [b'0591'],
        [b'020']], dtype=object)>,
 'ad_code': <tf.Tensor: id=38794, shape=(1024, 1), dtype=string, numpy=
 array(

In [66]:
labels

{'click_labels': <tf.Tensor: id=38987, shape=(1024, 1), dtype=float32, numpy=
 array([[0.],
        [0.],
        [0.],
        ...,
        [0.],
        [0.],
        [0.]], dtype=float32)>,
 'order_labels': <tf.Tensor: id=38989, shape=(1024, 1), dtype=float32, numpy=
 array([[0.],
        [0.],
        [0.],
        ...,
        [0.],
        [0.],
        [0.]], dtype=float32)>,
 'gmv_labels': <tf.Tensor: id=38988, shape=(1024, 1), dtype=float32, numpy=
 array([[0.],
        [0.],
        [0.],
        ...,
        [0.],
        [0.],
        [0.]], dtype=float32)>}

In [76]:
reshaped_features = {}

for feature_info in get_features_info():
    # [batch_size, 1] -> [batch_size, 1]
    reshaped_features[feature_info.feature_name] = tf.reshape(features[feature_info.feature_name], [batch_size, 1])

In [78]:
# reshaped_features

In [79]:
"""PLE"""
"""
在 TensorFlow Estimator API 中，model_fn 是一个函数，用于定义模型的结构和行为，并根据不同的模式返回不同的 tf.estimator.EstimatorSpec 对象。
model_fn 的输入参数包括 features、labels 和 mode，其中 mode 是一个枚举值，用于指示当前的运行模式。

在 model_fn 中，通常会根据 mode 的值来判断当前的运行模式，并执行相应的逻辑。虽然代码的顺序不会影响模型的行为（因为 mode 的值决定了执行哪一段代码），但为了代码的可读性和逻辑清晰性，通常会按照以下顺序编写：
    tf.estimator.ModeKeys.PREDICT
    tf.estimator.ModeKeys.TRAIN
    tf.estimator.ModeKeys.EVAL
"""
def model_fn(features, labels, mode, params, config):
    logger.warning(f'model_fn params: {params}')
    
    hidden_dims = params['hidden_dims']
    feature_columns = params['feature_columns']
    alpha = params['alpha']

    # print("111222333",feature_columns)
    
    """
    feature_columns：
        feature_columns 是一个列表，其中每个元素定义了输入特征的类型和处理方式。tf.keras.layers.DenseFeatures 层会根据这些定义将输入特征（如 reshaped_features）转换为密集张量。
    [
        EmbeddingColumn(categorical_column=VocabularyListCategoricalColumn(key='licensetag', vocabulary_list=('沪牌', '深牌', '粤A牌', '', '京牌', '外牌'), dtype=tf.string, default_value=-1, num_oov_buckets=0), dimension=9, combiner='mean', initializer=<tensorflow.python.ops.init_ops.TruncatedNormal object at 0x7fa06466ffd0>, ckpt_to_load_from=None, tensor_name_in_ckpt=None, max_norm=None, trainable=True), 
        EmbeddingColumn(categorical_column=VocabularyListCategoricalColumn(key='platform', vocabulary_list=('others', 'mPaaS', 'Android', 'iOS', 'WeChat', 'H5', 'Alipay'), dtype=tf.string, default_value=-1, num_oov_buckets=0), dimension=10, combiner='mean', initializer=<tensorflow.python.ops.init_ops.TruncatedNormal object at 0x7fa06466f150>, ckpt_to_load_from=None, tensor_name_in_ckpt=None, max_norm=None, trainable=True), 
        EmbeddingColumn(categorical_column=VocabularyListCategoricalColumn(key='city_code', vocabulary_list=('0836', '0809', '0760', '0798', '0535', '0514', '0737', '0379', '0393', '0802', '0370', '0733', '0993',
    ]
    """
    example_name = 'car_name'  
    batch_size = tf.shape(input=features[example_name])[0] # 360
    
    n_task = 3
    n_experts = [4,4,4]
    n_expert_share = 4
    expert_dim = 32
    dnn_reg_l2 = 0.0005  # 1e-6

    dnn_hidden_units = (512,128)
    drop_rate = 0.1
    targets = ['click_labels','order_labels','gmv_labels']

    reshaped_features = {}

    for feature_info in get_features_info():
        # [batch_size, 1] -> [batch_size, 1]
        reshaped_features[feature_info.feature_name] = tf.reshape(features[feature_info.feature_name], [batch_size, 1])

    
    """
    tf.keras.layers.DenseFeatures 是 TensorFlow 提供的一个 Keras 层，用于将特征列（feature_columns）转换为密集张量（dense tensor）。
    
    输入前格式：
        reshaped_features = {
            'feature1': [[1.0], [2.0], [3.0]],
            'feature2': [['A'], ['B'], ['C']]
        }
    
    数值特征列（numeric_column）
        对于数值特征（如 feature1），可以使用 tf.feature_column.numeric_column 定义。
        numeric_feature_column = tf.feature_column.numeric_column(key='feature1')
        key：特征的名称，必须与 reshaped_features 中的键一致。
        数值特征列会直接将输入的数值数据转换为密集张量。
    
    分类特征列（categorical_column_with_vocabulary_list）
        对于分类特征（如 feature2），可以使用 tf.feature_column.categorical_column_with_vocabulary_list 定义：
        categorical_feature_column = tf.feature_column.categorical_column_with_vocabulary_list(key='feature2',vocabulary_list=['A', 'B', 'C'])
        key：特征的名称，必须与 reshaped_features 中的键一致。
        vocabulary_list：分类特征的所有可能取值列表。
        分类特征列会将输入的分类数据映射为整数 ID（如 A -> 0, B -> 1, C -> 2）。
    
    嵌入特征列（embedding_column）：分类特征是指那些取值为有限类别（而非连续数值）的特征，例如性别（男/女）、国家（中国、美国、日本等）、用户类型（新用户、老用户等）
        如果分类特征的维度较高，通常会将其转换为嵌入向量（embedding）。可以使用 tf.feature_column.embedding_column 包装分类特征列：
        embedding_feature_column = tf.feature_column.embedding_column(categorical_column=categorical_feature_column,dimension=8  # 嵌入向量的维度)
        dimension：嵌入向量的维度，可以根据需要调整。
    """
    x = tf.keras.layers.DenseFeatures(feature_columns)(reshaped_features) # (?, 1368)
    
    for num in dnn_hidden_units:
        input_embed = Dropout(drop_rate)(Dense(num,activation = 'relu', kernel_regularizer=regularizers.l2(dnn_reg_l2))(x)) # (?, 128)
    
    # 添加BN层
    bn = tf.layers.BatchNormalization(axis=-1, center=True, scale=True, trainable=True)
    input_embed_bn = bn(input_embed) # (?, 128)
    # 调用ple模型
    towers = PleLayer(n_task,n_experts,expert_dim,n_expert_share)(input_embed_bn)
    # 输出每个任务的output   shape=(?, 1)  # outputs[0]  click pred ; outputs[1] order pred
    # 分类任务：是否点击/下单，二分类
    outputs_cls = [Dense(1,activation = 'sigmoid',kernel_regularizer=regularizers.l2(dnn_reg_l2), name = f,use_bias = True)(_t) for f,_t in zip(['click_labels','order_labels'],towers)]
    # 回归任务：gmv
    outputs_reg = [Dense(1,activation = 'relu',kernel_regularizer=regularizers.l2(dnn_reg_l2), name = f,use_bias = True)(_t) for f,_t in zip(['gmv_labels'],towers)]
    
    click_pre = outputs_cls[0]
    order_pre = outputs_cls[1]
    gmv_pre = outputs_reg[0]
    
    y = order_pre
    threshold = 0.5  # 标签阈值设置
    one = tf.ones_like(y)  # 生成与y大小一致的值全部为1的矩阵
    zero = tf.zeros_like(y)
    pred = tf.where(y < threshold, x=zero, y=one)  # 数值小于0.5置0，大于0.5置1

    predictions = {
        "click_pre": click_pre,
        "order_pre": order_pre,
        "gmv_pre": gmv_pre,
        "classes": pred,
        "y": y
    }

    
    """
    tf.estimator.ModeKeys.PREDICT 是 TensorFlow Estimator API 中的一个枚举值，表示模型当前处于预测模式。在预测模式下，模型的目的是根据输入特征生成预测结果，而不是进行训练或评估。
    tf.estimator.EstimatorSpec 是一个类，用于封装模型的输出和行为。它是一个通用的返回值，适用于 Estimator API 中的 model_fn（模型函数）。在不同的模式下（如训练、评估、预测），EstimatorSpec 的参数会有所不同。
    
    mode=mode：指定当前模式为 PREDICT，告诉 Estimator 当前正在执行预测操作。
    predictions=y：
        predictions 参数用于指定预测结果。
        y 是模型生成的预测输出，通常是一个张量。
    
    """
    if mode == tf.estimator.ModeKeys.PREDICT:
        return tf.estimator.EstimatorSpec(mode=mode, predictions=y)  # [batch_size, 2]
    
    """BinaryCrossentropy"""
    bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)
    click_cross_entropy = tf.reduce_mean(bce(labels['click_labels'], click_pre)) 
    order_cross_entropy = tf.reduce_mean(bce(labels['order_labels'], order_pre))
    gmv_mse = tf.losses.mean_squared_error(labels['gmv_labels'], gmv_pre)

    
    print("成功")
    """
    损失函数：
        1、三个任务直接相加
        2、三个任务手动设置权重：点击*0.2 + 下单*0.6 + gmv*0.2
        3、三个任务通过线性规划进行动态设置权重
    """
    # 手动设置权重
    # loss1 = 0.2*click_cross_entropy + 0.6*order_cross_entropy + 0.2*gmv_mse
    loss1 = 0.2*click_cross_entropy + 0.8*order_cross_entropy
    
    tf.summary.scalar('loss1', click_cross_entropy)
    tf.summary.scalar('loss2', order_cross_entropy)
    tf.summary.scalar('loss3', gmv_mse)
    
    
    """
    用于定义模型在评估模式（EVAL）下的行为。它计算了模型在评估数据上的损失和多个评估指标，并将这些结果封装到 tf.estimator.EstimatorSpec 中返回。
    tf.estimator.ModeKeys.EVAL 是一个枚举值，表示模型当前处于评估模式。在评估模式下，模型的目的是计算损失和评估指标，以衡量模型的性能。
    eval_metric_ops 是一个字典，用于定义评估模式下需要计算的评估指标。每个键是一个指标的名称，值是一个指标的计算结果。
    返回一个 tf.estimator.EstimatorSpec 对象，封装了评估模式下的损失和评估指标。
    """
    if mode == tf.estimator.ModeKeys.EVAL:
        eval_metric_ops = {
            'click_metric/auc': auc(labels['click_labels'], click_pre),
            'order_metric/auc': auc(labels['order_labels'], order_pre),
            'gmv_metric/mse': mse(labels['gmv_labels'], gmv_pre),
        }
        return tf.estimator.EstimatorSpec(mode=mode, loss=loss1, eval_metric_ops=eval_metric_ops)

    
    """
    用于定义模型在训练模式（TRAIN）下的行为。它主要负责设置优化器、定义训练操作，并返回一个 tf.estimator.EstimatorSpec 对象。
    tf.estimator.ModeKeys.TRAIN 是一个枚举值，表示模型当前处于训练模式。在训练模式下，模型的目的是通过优化损失函数来更新模型参数。
    optimizer.minimize：调用优化器的 minimize 方法，定义训练操作。
    loss=loss1：指定需要最小化的损失函数。loss1 是模型的损失值，通常是预测值与真实值之间的差异。
    global_step=tf.train.get_global_step()：global_step 是一个 TensorFlow 中的全局步数计数器，用于记录训练过程中的迭代次数。它在保存模型和学习率衰减等操作中非常有用。
    
    """
    if mode == tf.estimator.ModeKeys.TRAIN:
        optimizer = tf.train.AdamOptimizer(learning_rate=3e-5) # 3e-4
        train_op = optimizer.minimize(
            loss=loss1,
            global_step=tf.train.get_global_step()
        )
        return tf.estimator.EstimatorSpec(mode=mode, loss=loss1, train_op=train_op)

    raise ValueError('mode={} unrecognized'.format(mode))

In [80]:
def main(unused_argv):
    """参数设置""" 
    # estimator_params = param_utils.get_object_from_input_param('estimator_params')
    # train_partition = param_utils.get_object_from_input_param('train_partition')
    # eval_partition = param_utils.get_object_from_input_param('eval_partition')
    # max_steps = param_utils.get_object_from_input_param('max_steps')
    # batch_size = param_utils.get_object_from_input_param('batch_size')
    
    max_steps = 50000   # 200000
    batch_size = 32

    """一天训练一天预测"""
#     train_partition = [['pt=20241222']]
#     eval_partition = [['pt=20241224']]
#     estimator_params = {
#         'hidden_dims': [128, 64, 32],
#         'feature_columns': get_feature_columns(['20241222','20241224']),
#         'alpha': 0.5
#     }
    
    
    date_converter = date_convert_utils.DateConvertUtils()
    train_pt = date_converter.parse_data_date("${yyyymmdd-2}")
    eval_pt = date_converter.parse_data_date("${yyyymmdd-1}")
    train_partition = [['pt=' + str(train_pt)]] # [['pt=20240709']]
    eval_partition = [['pt=' + str(eval_pt)]] # [['pt=20240710']]
    
    print("训练数据pt:",train_partition)
    print("测试数据pt:",eval_partition)
    
    estimator_params = {
        'hidden_dims': [128, 64, 32],
        'feature_columns': get_feature_columns([train_pt, eval_pt]), # '20240709','20240710'
        'alpha': 0.5
    }
    
    
    seed = 2024
    tf.set_random_seed(seed)  # 为TensorFlow设置全局随机种子

    train_table = train_data_df
    eval_table = train_data_df

    config, _ = tf_context.get_tf_config(
        save_checkpoints_secs=None,
        save_checkpoints_steps=500,
        keep_checkpoint_max=5
    )

    ranker = tf.estimator.Estimator(
        model_fn=model_fn,
        model_dir=config.model_dir,
        config=config,
        params=estimator_params
    )

    train_spec = tf.estimator.TrainSpec(input_fn=lambda: input_fn(train_table, train_partition, batch_size=batch_size), max_steps=max_steps)
    eval_spec = tf.estimator.EvalSpec(input_fn=lambda: input_fn(eval_table, eval_partition, num_epochs=1, batch_size=batch_size))

    print("result 开始")
    tf.estimator.train_and_evaluate(ranker, train_spec, eval_spec)
    print("result 结束")
    
    
    return

In [81]:
if __name__ == '__main__':
    tf.app.run()

训练数据pt: [['pt=20250120']]
测试数据pt: [['pt=20250121']]


W0122 15:07:26.493920 140342019123008 tf_context.py:75] model will be stored in path: public/lizhiqiang195/5946941


INFO:tensorflow:Using config: {'_model_dir': 'public/lizhiqiang195/5946941', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': 500, '_save_checkpoints_secs': None, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_service': None, '_cluster_spec': <tensorflow.python.training.server_lib.ClusterSpec object at 0x7fa34e86b150>, '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}


I0122 15:07:26.496703 140342019123008 estimator.py:212] Using config: {'_model_dir': 'public/lizhiqiang195/5946941', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': 500, '_save_checkpoints_secs': None, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_service': None, '_cluster_spec': <tensorflow.python.training.server_lib.ClusterSpec object at 0x7fa34e86b150>, '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}


result 开始
INFO:tensorflow:Not using Distribute Coordinator.


...

111222333 [EmbeddingColumn(categorical_column=VocabularyListCategoricalColumn(key='licensetag', vocabulary_list=('沪牌', '深牌', '粤A牌', '', '京牌', '外牌'), dtype=tf.string, default_value=-1, num_oov_buckets=0), dimension=9, combiner='mean', initializer=<tensorflow.python.ops.init_ops.TruncatedNormal object at 0x7fa06466ffd0>, ckpt_to_load_from=None, tensor_name_in_ckpt=None, max_norm=None, trainable=True), EmbeddingColumn(categorical_column=VocabularyListCategoricalColumn(key='platform', vocabulary_list=('others', 'mPaaS', 'Android', 'iOS', 'WeChat', 'H5', 'Alipay'), dtype=tf.string, default_value=-1, num_oov_buckets=0), dimension=10, combiner='mean', initializer=<tensorflow.python.ops.init_ops.TruncatedNormal object at 0x7fa06466f150>, ckpt_to_load_from=None, tensor_name_in_ckpt=None, max_norm=None, trainable=True), EmbeddingColumn(categorical_column=VocabularyListCategoricalColumn(key='city_code', vocabulary_list=('0836', '0809', '0760', '0798', '0535', '0514', '0737', '0379', '0393', '0802

I0122 15:07:50.931324 140342019123008 estimator.py:1150] Done calling model_fn.


INFO:tensorflow:Create CheckpointSaverHook.


I0122 15:07:50.935248 140342019123008 basic_session_run_hooks.py:541] Create CheckpointSaverHook.


KeyboardInterrupt: 